# Comparison against XChainWatcher

### Test 1: Full removal of bridge cases (no training leakage)

In [217]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender")  # adjust if notebook isn't at repo root

# label -> inference run folder
RUNS = {
    "XChainWatcher (single-chain)": BASE_DIR / "outputs/20260716-single-internaltx-comparison-xchainwatcher-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607171519/inference_xchainwatcher-test-ronin1-nomad-recall_202607171522",
    "XScope (single-chain)": BASE_DIR / "outputs/20260720-single-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607201714/inference_comparison-xscope-excludeall_202607211134",
    "XScope (cross-chain)": BASE_DIR / "outputs/20260721-cctx-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607211121/inference_comparison-xscope-fullexclusion_202607211627",
}

METRIC_COLS = [
    "loss", "accuracy",
    "precision_normal", "recall_normal", "f1_normal",
    "precision_anomaly", "recall_anomaly", "f1_anomaly", "f2_anomaly",
    "roc_auc", "pr_auc", "mcc",
]


In [218]:
def load_eval_stats(run_name: str, run_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(run_dir / "eval_stats.csv")
    df.insert(0, "test", run_name)
    return df

all_stats = pd.concat(
    [load_eval_stats(name, path) for name, path in RUNS.items()],
    ignore_index=True,
)
all_stats


,test,fold,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain),fold_1,1.424583,0.769126,0.769126,1.000000,0.869498,0.000000,0.000000,0.000000,0.000000,0.586881,0.255674,0.000000
1,XChainWatcher (single-chain),fold_2,0.585240,0.728142,0.961929,0.673179,0.792059,0.455621,0.911243,0.607495,0.759369,0.714368,0.365168,0.493989
2,XChainWatcher (single-chain),fold_3,1.054961,0.789617,0.785216,1.000000,0.879687,1.000000,0.088757,0.163043,0.108538,0.674893,0.365832,0.263996
3,XChainWatcher (single-chain),fold_4,0.927721,0.771858,0.771233,1.000000,0.870843,1.000000,0.011834,0.023392,0.014749,0.674787,0.367592,0.095535
4,XChainWatcher (single-chain),fold_5,1.293298,0.769126,0.769126,1.000000,0.869498,0.000000,0.000000,0.000000,0.000000,0.533942,0.334010,0.000000
5,XScope (single-chain),fold_1,7.154039,0.044444,0.044444,1.000000,0.085106,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
6,XScope (single-chain),fold_2,9.348442,0.044444,0.044444,1.000000,0.085106,0.000000,0.000000,0.000000,0.000000,0.953488,0.997909,0.000000
7,XScope (single-chain),fold_3,9.332342,0.044444,0.044444,1.000000,0.085106,0.000000,0.000000,0.000000,0.000000,0.651163,0.981517,0.000000
8,XScope (single-chain),fold_4,6.863031,0.044444,0.044444,1.000000,0.085106,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
9,XScope (single-chain),fold_5,10.360770,0.044444,0.044444,1.000000,0.085106,0.000000,0.000000,0.000000,0.000000,0.651163,0.981484,0.000000


In [219]:
summary = all_stats.groupby("test")[METRIC_COLS].agg(["mean", "var"])
summary

loss            accuracy            \
                                  mean       var      mean       var   
test                                                                   
XChainWatcher (single-chain)  1.057161  0.107557  0.765574  0.000511   
XScope (cross-chain)          5.526787  3.743696  0.229630  0.010562   
XScope (single-chain)         8.611725  2.325995  0.044444  0.000000   

                             precision_normal           recall_normal  \
                                         mean       var          mean   
test                                                                    
XChainWatcher (single-chain)         0.811326  0.007133      0.934636   
XScope (cross-chain)                 0.127656  0.000297      1.000000   
XScope (single-chain)                0.044444  0.000000      1.000000   

                                       f1_normal            ... f1_anomaly  \
                                   var      mean       var  ...       mean   
test                                                        ...              
XChainWatcher (single-chain)  0.021362  0.856317  0.001308  ...   0.158786   
XScope (cross-chain)          0.000000  0.226083  0.000709  ...   0.221983   
XScope (single-chain)         0.000000  0.085106  0.000000  ...   0.000000   

                                       f2_anomaly             roc_auc  \
                                   var       mean       var      mean   
test                                                                    
XChainWatcher (single-chain)  0.067529   0.176531  0.108206  0.636974   
XScope (cross-chain)          0.026684   0.158352  0.017248  0.505556   
XScope (single-chain)         0.000000   0.000000  0.000000  0.851163   

                                          pr_auc                 mcc            
                                   var      mean       var      mean       var  
test                                                                            
XChainWatcher (single-chain)  0.005496  0.337655  0.002295  0.170704  0.044287  
XScope (cross-chain)          0.113291  0.903558  0.004159  0.124751  0.003813  
XScope (single-chain)         0.033694  0.992182  0.000096  0.000000  0.000000  

[3 rows x 24 columns]

In [220]:
summary_flat = summary.copy()
summary_flat.columns = [f"{metric}_{stat}" for metric, stat in summary_flat.columns]
summary_flat = summary_flat.reindex(RUNS.keys())  # preserve the order defined in RUNS
summary_flat = summary_flat.reset_index()
summary_flat


,test,loss_mean,loss_var,accuracy_mean,accuracy_var,precision_normal_mean,precision_normal_var,recall_normal_mean,recall_normal_var,f1_normal_mean,...,f1_anomaly_mean,f1_anomaly_var,f2_anomaly_mean,f2_anomaly_var,roc_auc_mean,roc_auc_var,pr_auc_mean,pr_auc_var,mcc_mean,mcc_var
0,XChainWatcher (single-chain),1.057161,0.107557,0.765574,0.000511,0.811326,0.007133,0.934636,0.021362,0.856317,...,0.158786,0.067529,0.176531,0.108206,0.636974,0.005496,0.337655,0.002295,0.170704,0.044287
1,XScope (single-chain),8.611725,2.325995,0.044444,0.000000,0.044444,0.000000,1.000000,0.000000,0.085106,...,0.000000,0.000000,0.000000,0.000000,0.851163,0.033694,0.992182,0.000096,0.000000,0.000000
2,XScope (cross-chain),5.526787,3.743696,0.229630,0.010562,0.127656,0.000297,1.000000,0.000000,0.226083,...,0.221983,0.026684,0.158352,0.017248,0.505556,0.113291,0.903558,0.004159,0.124751,0.003813


In [221]:
# latex_table = summary_flat.round(4).to_latex(
#     index=False,
#     escape=True,
#     caption="Mean and variance of evaluation metrics per test.",
#     label="tab:eval-summary",
# )
# print(latex_table)


In [222]:
def fmt_mean_var(mean, var, decimals=4, percent=False):
    if percent:
        mean *= 100
        var *= 100
    suffix = "\\%" if percent else ""
    return f"{mean:05.{decimals}f} ± {var:05.{decimals}f}{suffix}"

combined = pd.DataFrame({"test": summary_flat["test"]})
for metric in METRIC_COLS:
    combined[metric] = [
        fmt_mean_var(m, v, decimals=2 if metric not in ["loss"] else 4, percent=True if metric not in ["loss"] else False)
        for m, v in zip(summary_flat[f"{metric}_mean"], summary_flat[f"{metric}_var"])
    ]

combined


,test,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain),1.0572 ± 0.1076,76.56 ± 00.05\%,81.13 ± 00.71\%,93.46 ± 02.14\%,85.63 ± 00.13\%,49.11 ± 25.04\%,20.24 ± 15.84\%,15.88 ± 06.75\%,17.65 ± 10.82\%,63.70 ± 00.55\%,33.77 ± 00.23\%,17.07 ± 04.43\%
1,XScope (single-chain),8.6117 ± 2.3260,04.44 ± 00.00\%,04.44 ± 00.00\%,100.00 ± 00.00\%,08.51 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,85.12 ± 03.37\%,99.22 ± 00.01\%,00.00 ± 00.00\%
2,XScope (cross-chain),5.5268 ± 3.7437,22.96 ± 01.06\%,12.77 ± 00.03\%,100.00 ± 00.00\%,22.61 ± 00.07\%,100.00 ± 00.00\%,13.33 ± 01.34\%,22.20 ± 02.67\%,15.84 ± 01.72\%,50.56 ± 11.33\%,90.36 ± 00.42\%,12.48 ± 00.38\%


In [223]:
import yaml

def load_dataset_stats(run_dir: Path) -> dict:
    with open(run_dir / "params.yaml") as f:
        params = yaml.safe_load(f)
    stats = params["dataset"]["stats"]
    return {
        "total_samples": stats["total_samples"],
        "anomaly_samples": stats["anomaly_samples"],
    }

dataset_stats = pd.DataFrame(
    [{"test": name, **load_dataset_stats(path)} for name, path in RUNS.items()]
)

combined = combined.merge(dataset_stats, on="test")
cols = ["test", "total_samples", "anomaly_samples"] + METRIC_COLS
combined = combined[cols]
combined


,test,total_samples,anomaly_samples,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain),732,169,1.0572 ± 0.1076,76.56 ± 00.05\%,81.13 ± 00.71\%,93.46 ± 02.14\%,85.63 ± 00.13\%,49.11 ± 25.04\%,20.24 ± 15.84\%,15.88 ± 06.75\%,17.65 ± 10.82\%,63.70 ± 00.55\%,33.77 ± 00.23\%,17.07 ± 04.43\%
1,XScope (single-chain),45,43,8.6117 ± 2.3260,04.44 ± 00.00\%,04.44 ± 00.00\%,100.00 ± 00.00\%,08.51 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,85.12 ± 03.37\%,99.22 ± 00.01\%,00.00 ± 00.00\%
2,XScope (cross-chain),27,24,5.5268 ± 3.7437,22.96 ± 01.06\%,12.77 ± 00.03\%,100.00 ± 00.00\%,22.61 ± 00.07\%,100.00 ± 00.00\%,13.33 ± 01.34\%,22.20 ± 02.67\%,15.84 ± 01.72\%,50.56 ± 11.33\%,90.36 ± 00.42\%,12.48 ± 00.38\%


In [224]:
FILTERED_METRIC_COLS = ["precision_anomaly", "recall_anomaly", "f1_anomaly"]
combined_filtered = combined.copy()[["test", "total_samples", "anomaly_samples"] + FILTERED_METRIC_COLS]

column_headers = {
    "test": "Reference model (dataset type)",
    "total_samples": "\\#S",
    "anomaly_samples": "\\#A",
    "precision_anomaly": "Precision (anomaly)",
    "recall_anomaly": "Recall (anomaly)",
    "f1_anomaly": "F1 (anomaly)",
}

combined_filtered = combined_filtered.rename(columns=column_headers)
combined_filtered

,Reference model (dataset type),\#S,\#A,Precision (anomaly),Recall (anomaly),F1 (anomaly)
0,XChainWatcher (single-chain),732,169,49.11 ± 25.04\%,20.24 ± 15.84\%,15.88 ± 06.75\%
1,XScope (single-chain),45,43,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%
2,XScope (cross-chain),27,24,100.00 ± 00.00\%,13.33 ± 01.34\%,22.20 ± 02.67\%


In [225]:
combined_latex_ready = combined_filtered.copy()
for metric in FILTERED_METRIC_COLS:
    combined_latex_ready[column_headers[metric]] = combined_latex_ready[column_headers[metric]].str.replace(
        "±", r"$\pm$", regex=False
    )

latex_combined = combined_latex_ready.to_latex(
    index=False,
    escape=False,  # needed so $\pm$ isn't escaped into literal text
    caption="Leak-free agreement test metrics based on anomalies identified by prior works.",
    label="table:evaluation-leak-free-agreement",
)
print(latex_combined)


\begin{table}
\caption{Leak-free agreement test metrics based on anomalies identified by prior works.}
\label{table:evaluation-leak-free-agreement}
\begin{tabular}{lrrlll}
\toprule
Reference model (dataset type) & \#S & \#A & Precision (anomaly) & Recall (anomaly) & F1 (anomaly) \\
\midrule
XChainWatcher (single-chain) & 732 & 169 & 49.11 $\pm$ 25.04\% & 20.24 $\pm$ 15.84\% & 15.88 $\pm$ 06.75\% \\
XScope (single-chain) & 45 & 43 & 00.00 $\pm$ 00.00\% & 00.00 $\pm$ 00.00\% & 00.00 $\pm$ 00.00\% \\
XScope (cross-chain) & 27 & 24 & 100.00 $\pm$ 00.00\% & 13.33 $\pm$ 01.34\% & 22.20 $\pm$ 02.67\% \\
\bottomrule
\end{tabular}
\end{table}



### Test 2: Leaky testing of bridge cases (stratified 50/50 split on training and testing sets)

In [226]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender")  # adjust if notebook isn't at repo root

# label -> inference run folder
RUNS = {
    "XChainWatcher (single-chain)": BASE_DIR / "outputs/20260716-single-internaltx-comparison-xchainwatcher-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_50label_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607191656/inference_xchainwatcher-test-ronin1-nomad-recall_202607191701",
    "XScope (single-chain)": BASE_DIR / "outputs/20260720-single-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_50label_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607211204/inference_comparison-xscope-50label_202607211220",
    "XScope (cross-chain)": BASE_DIR / "outputs/20260721-cctx-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_50label_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607211630/inference_cctx-xscope-50label_202607221002",
}

METRIC_COLS = [
    "loss", "accuracy",
    "precision_normal", "recall_normal", "f1_normal",
    "precision_anomaly", "recall_anomaly", "f1_anomaly", "f2_anomaly",
    "roc_auc", "pr_auc", "mcc",
]


In [227]:
def load_eval_stats(run_name: str, run_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(run_dir / "eval_stats.csv")
    df.insert(0, "test", run_name)
    return df

all_stats = pd.concat(
    [load_eval_stats(name, path) for name, path in RUNS.items()],
    ignore_index=True,
)
all_stats


,test,fold,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain),fold_1,0.082451,0.967123,0.99631,0.960854,0.978261,0.882979,0.988095,0.932584,0.965116,0.998475,0.995254,0.913455
1,XChainWatcher (single-chain),fold_2,0.172952,0.967123,0.99631,0.960854,0.978261,0.882979,0.988095,0.932584,0.965116,0.976487,0.918267,0.913455
2,XChainWatcher (single-chain),fold_3,0.089041,0.967123,0.99631,0.960854,0.978261,0.882979,0.988095,0.932584,0.965116,0.997543,0.989652,0.913455
3,XChainWatcher (single-chain),fold_4,0.073228,0.967123,0.99631,0.960854,0.978261,0.882979,0.988095,0.932584,0.965116,0.999110,0.997269,0.913455
4,XChainWatcher (single-chain),fold_5,0.098138,0.967123,0.99631,0.960854,0.978261,0.882979,0.988095,0.932584,0.965116,0.986612,0.949739,0.913455
5,XScope (single-chain),fold_1,0.032278,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,XScope (single-chain),fold_2,0.017472,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,XScope (single-chain),fold_3,0.016549,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
8,XScope (single-chain),fold_4,0.019140,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,XScope (single-chain),fold_5,0.411332,0.863636,0.25000,1.000000,0.400000,1.000000,0.857143,0.923077,0.882353,1.000000,1.000000,0.462910


In [228]:
summary = all_stats.groupby("test")[METRIC_COLS].agg(["mean", "var"])
summary

loss            accuracy            \
                                  mean       var      mean       var   
test                                                                   
XChainWatcher (single-chain)  0.103162  0.001605  0.967123  0.000000   
XScope (cross-chain)          0.094267  0.013180  0.969231  0.001775   
XScope (single-chain)         0.099354  0.030456  0.972727  0.003719   

                             precision_normal         recall_normal       \
                                         mean     var          mean  var   
test                                                                       
XChainWatcher (single-chain)          0.99631  0.0000      0.960854  0.0   
XScope (cross-chain)                  0.60000  0.3000      0.600000  0.3   
XScope (single-chain)                 0.85000  0.1125      1.000000  0.0   

                             f1_normal         ... f1_anomaly            \
                                  mean    var  ...       mean       var   
test                                           ...                        
XChainWatcher (single-chain)  0.978261  0.000  ...   0.932584  0.000000   
XScope (cross-chain)          0.600000  0.300  ...   0.984000  0.000480   
XScope (single-chain)         0.880000  0.072  ...   0.984615  0.001183   

                             f2_anomaly             roc_auc            \
                                   mean       var      mean       var   
test                                                                    
XChainWatcher (single-chain)   0.965116  0.000000  0.991645  0.000098   
XScope (cross-chain)           0.993443  0.000081  1.000000  0.000000   
XScope (single-chain)          0.976471  0.002768  1.000000  0.000000   

                                pr_auc                     mcc            
                                  mean           var      mean       var  
test                                                                      
XChainWatcher (single-chain)  0.970036  1.213592e-03  0.913455  0.000000  
XScope (cross-chain)          1.000000  0.000000e+00  0.600000  0.300000  
XScope (single-chain)         1.000000  1.232595e-32  0.892582  0.057693  

[3 rows x 24 columns]

In [229]:
summary_flat = summary.copy()
summary_flat.columns = [f"{metric}_{stat}" for metric, stat in summary_flat.columns]
summary_flat = summary_flat.reindex(RUNS.keys())  # preserve the order defined in RUNS
summary_flat = summary_flat.reset_index()
summary_flat


,test,loss_mean,loss_var,accuracy_mean,accuracy_var,precision_normal_mean,precision_normal_var,recall_normal_mean,recall_normal_var,f1_normal_mean,...,f1_anomaly_mean,f1_anomaly_var,f2_anomaly_mean,f2_anomaly_var,roc_auc_mean,roc_auc_var,pr_auc_mean,pr_auc_var,mcc_mean,mcc_var
0,XChainWatcher (single-chain),0.103162,0.001605,0.967123,0.000000,0.99631,0.0000,0.960854,0.0,0.978261,...,0.932584,0.000000,0.965116,0.000000,0.991645,0.000098,0.970036,1.213592e-03,0.913455,0.000000
1,XScope (single-chain),0.099354,0.030456,0.972727,0.003719,0.85000,0.1125,1.000000,0.0,0.880000,...,0.984615,0.001183,0.976471,0.002768,1.000000,0.000000,1.000000,1.232595e-32,0.892582,0.057693
2,XScope (cross-chain),0.094267,0.013180,0.969231,0.001775,0.60000,0.3000,0.600000,0.3,0.600000,...,0.984000,0.000480,0.993443,0.000081,1.000000,0.000000,1.000000,0.000000e+00,0.600000,0.300000


In [230]:
# latex_table = summary_flat.round(4).to_latex(
#     index=False,
#     escape=True,
#     caption="Mean and variance of evaluation metrics per test.",
#     label="tab:eval-summary",
# )
# print(latex_table)


In [231]:
def fmt_mean_var(mean, var, decimals=4, percent=False):
    if percent:
        mean *= 100
        var *= 100
    suffix = "\\%" if percent else ""
    return f"{mean:05.{decimals}f} ± {var:05.{decimals}f}{suffix}"

combined = pd.DataFrame({"test": summary_flat["test"]})
for metric in METRIC_COLS:
    combined[metric] = [
        fmt_mean_var(m, v, decimals=2 if metric not in ["loss"] else 4, percent=True if metric not in ["loss"] else False)
        for m, v in zip(summary_flat[f"{metric}_mean"], summary_flat[f"{metric}_var"])
    ]

combined

,test,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain),0.1032 ± 0.0016,96.71 ± 00.00\%,99.63 ± 00.00\%,96.09 ± 00.00\%,97.83 ± 00.00\%,88.30 ± 00.00\%,98.81 ± 00.00\%,93.26 ± 00.00\%,96.51 ± 00.00\%,99.16 ± 00.01\%,97.00 ± 00.12\%,91.35 ± 00.00\%
1,XScope (single-chain),0.0994 ± 0.0305,97.27 ± 00.37\%,85.00 ± 11.25\%,100.00 ± 00.00\%,88.00 ± 07.20\%,100.00 ± 00.00\%,97.14 ± 00.41\%,98.46 ± 00.12\%,97.65 ± 00.28\%,100.00 ± 00.00\%,100.00 ± 00.00\%,89.26 ± 05.77\%
2,XScope (cross-chain),0.0943 ± 0.0132,96.92 ± 00.18\%,60.00 ± 30.00\%,60.00 ± 30.00\%,60.00 ± 30.00\%,96.92 ± 00.18\%,100.00 ± 00.00\%,98.40 ± 00.05\%,99.34 ± 00.01\%,100.00 ± 00.00\%,100.00 ± 00.00\%,60.00 ± 30.00\%


In [232]:
import yaml

def load_dataset_stats(run_dir: Path) -> dict:
    with open(run_dir / "params.yaml") as f:
        params = yaml.safe_load(f)
    stats = params["dataset"]["stats"]
    return {
        "total_samples": stats["total_samples"],
        "anomaly_samples": stats["anomaly_samples"],
    }

dataset_stats = pd.DataFrame(
    [{"test": name, **load_dataset_stats(path)} for name, path in RUNS.items()]
)

combined = combined.merge(dataset_stats, on="test")
cols = ["test", "total_samples", "anomaly_samples"] + METRIC_COLS
combined = combined[cols]
combined


,test,total_samples,anomaly_samples,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain),365,84,0.1032 ± 0.0016,96.71 ± 00.00\%,99.63 ± 00.00\%,96.09 ± 00.00\%,97.83 ± 00.00\%,88.30 ± 00.00\%,98.81 ± 00.00\%,93.26 ± 00.00\%,96.51 ± 00.00\%,99.16 ± 00.01\%,97.00 ± 00.12\%,91.35 ± 00.00\%
1,XScope (single-chain),22,21,0.0994 ± 0.0305,97.27 ± 00.37\%,85.00 ± 11.25\%,100.00 ± 00.00\%,88.00 ± 07.20\%,100.00 ± 00.00\%,97.14 ± 00.41\%,98.46 ± 00.12\%,97.65 ± 00.28\%,100.00 ± 00.00\%,100.00 ± 00.00\%,89.26 ± 05.77\%
2,XScope (cross-chain),13,12,0.0943 ± 0.0132,96.92 ± 00.18\%,60.00 ± 30.00\%,60.00 ± 30.00\%,60.00 ± 30.00\%,96.92 ± 00.18\%,100.00 ± 00.00\%,98.40 ± 00.05\%,99.34 ± 00.01\%,100.00 ± 00.00\%,100.00 ± 00.00\%,60.00 ± 30.00\%


In [233]:
FILTERED_METRIC_COLS = ["precision_anomaly", "recall_anomaly", "f1_anomaly"]
combined_filtered = combined.copy()[["test", "total_samples", "anomaly_samples"] + FILTERED_METRIC_COLS]

column_headers = {
    "test": "Reference model (dataset type)",
    "total_samples": "\\#S",
    "anomaly_samples": "\\#A",
    "precision_anomaly": "Precision (anomaly)",
    "recall_anomaly": "Recall (anomaly)",
    "f1_anomaly": "F1 (anomaly)",
}

combined_filtered = combined_filtered.rename(columns=column_headers)
combined_filtered

,Reference model (dataset type),\#S,\#A,Precision (anomaly),Recall (anomaly),F1 (anomaly)
0,XChainWatcher (single-chain),365,84,88.30 ± 00.00\%,98.81 ± 00.00\%,93.26 ± 00.00\%
1,XScope (single-chain),22,21,100.00 ± 00.00\%,97.14 ± 00.41\%,98.46 ± 00.12\%
2,XScope (cross-chain),13,12,96.92 ± 00.18\%,100.00 ± 00.00\%,98.40 ± 00.05\%


In [234]:
combined_latex_ready = combined_filtered.copy()
for metric in FILTERED_METRIC_COLS:
    combined_latex_ready[column_headers[metric]] = combined_latex_ready[column_headers[metric]].str.replace(
        "±", r"$\pm$", regex=False
    )

latex_combined = combined_latex_ready.to_latex(
    index=False,
    escape=False,  # needed so $\pm$ isn't escaped into literal text
    caption="Upper-bound agreement test metrics based on known anomalies, identified by prior works.",
    label="table:evaluation-upper-bound-agreement",
)
print(latex_combined)


\begin{table}
\caption{Upper-bound agreement test metrics based on known anomalies, identified by prior works.}
\label{table:evaluation-upper-bound-agreement}
\begin{tabular}{lrrlll}
\toprule
Reference model (dataset type) & \#S & \#A & Precision (anomaly) & Recall (anomaly) & F1 (anomaly) \\
\midrule
XChainWatcher (single-chain) & 365 & 84 & 88.30 $\pm$ 00.00\% & 98.81 $\pm$ 00.00\% & 93.26 $\pm$ 00.00\% \\
XScope (single-chain) & 22 & 21 & 100.00 $\pm$ 00.00\% & 97.14 $\pm$ 00.41\% & 98.46 $\pm$ 00.12\% \\
XScope (cross-chain) & 13 & 12 & 96.92 $\pm$ 00.18\% & 100.00 $\pm$ 00.00\% & 98.40 $\pm$ 00.05\% \\
\bottomrule
\end{tabular}
\end{table}



### Test 3: Cross-testing in different bridge scenarios (test on a different incident each time, use the other incidents for training)

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender")  # adjust if notebook isn't at repo root

RUNS = {
    "XChainWatcher (single-chain comparison)": {
        "Ronin \\#1": BASE_DIR / "outputs/20260716-single-internaltx-comparison-xchainwatcher-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullronin1_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221016/inference_single-xchainwatcher-ronin1_202607221024",
        "Nomad": BASE_DIR / "outputs/20260716-single-internaltx-comparison-xchainwatcher-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullnomad_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221010/inference_single-xchainwatcher-fullnomad_202607221015",
    },
    "XScope (single-chain model)": {
        "Qubit": BASE_DIR / "outputs/20260720-single-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullqubit_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221032/inference_single-xscope-fullqubit_202607221211",
        "THORChain \\#1": BASE_DIR / "outputs/20260720-single-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullthorchain1_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221038/inference_single-xscope-fullthorchain1_202607221212",
        "THORChain \\#2": BASE_DIR / "outputs/20260720-single-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullthorchain2_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221045/inference_single-xscope-fullthorchain2_202607221213",
        "THORChain \\#3": BASE_DIR / "outputs/20260720-single-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullthorchain3_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221051/inference_single-xscope-thorchain3_202607221214"
    },
    "XScope (cross-chain model)": {
        "Qubit": BASE_DIR / "outputs/20260721-cctx-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullqubit_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221554/inference_cctx-xscope-fullqubit_202607221617",
        "THORChain \\#2": BASE_DIR / "outputs/20260721-cctx-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullthorchain2_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221104/inference_cctx-xscope-fullthorchain2_202607221216",
        "THORChain \\#3": BASE_DIR / "outputs/20260721-cctx-internaltx-comparison-xscope-ronin2-omni-nomad-polyn-qubit-harmony-thor123-meter-ronin1/train_fullthorchain3_epochs100_augment0_lr0.005_nosf_es15_ts0.0_202607221123/inference_cctx-xscope-fullthorchain3_202607221217"
    },
}

# flatten to (model, incident, path) preserving nested order — used by every later cell
RUN_ROWS = [
    (model, incident, path)
    for model, incidents in RUNS.items()
    for incident, path in incidents.items()
]

METRIC_COLS = [
    "loss", "accuracy",
    "precision_normal", "recall_normal", "f1_normal",
    "precision_anomaly", "recall_anomaly", "f1_anomaly", "f2_anomaly",
    "roc_auc", "pr_auc", "mcc",
]


<>:8: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<>:13: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<>:14: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<>:15: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<>:19: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<>:20: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<>:8: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will 

In [190]:
def load_eval_stats(model: str, incident: str, run_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(run_dir / "eval_stats.csv")
    df.insert(0, "incident", incident)
    df.insert(0, "model", model)
    return df

all_stats = pd.concat(
    [df for model, incident, path in RUN_ROWS
     if (df := load_eval_stats(model, incident, path)) is not None],
    ignore_index=True,
)
all_stats.head(10)

,model,incident,fold,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain comparison),Ronin #1,fold_1,4.304398,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,0.000000
1,XChainWatcher (single-chain comparison),Ronin #1,fold_2,9.200613,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,0.000000
2,XChainWatcher (single-chain comparison),Ronin #1,fold_3,8.048842,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,0.000000
3,XChainWatcher (single-chain comparison),Ronin #1,fold_4,15.940908,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.000000,0.000000
4,XChainWatcher (single-chain comparison),Ronin #1,fold_5,0.935105,0.500000,0.000000,0.000000,0.000000,1.000000,0.500000,0.666667,0.555556,NaN,1.000000,0.000000
5,XChainWatcher (single-chain comparison),Nomad,fold_1,0.269190,0.841096,0.853081,0.959147,0.903010,0.762887,0.443114,0.560606,0.483660,0.961158,0.811664,0.497775
6,XChainWatcher (single-chain comparison),Nomad,fold_2,0.743021,0.841096,0.853081,0.959147,0.903010,0.762887,0.443114,0.560606,0.483660,0.861254,0.634777,0.497775
7,XChainWatcher (single-chain comparison),Nomad,fold_3,0.822896,0.771233,0.771233,1.000000,0.870843,0.000000,0.000000,0.000000,0.000000,0.802587,0.581635,0.000000
8,XChainWatcher (single-chain comparison),Nomad,fold_4,0.421292,0.841096,0.853081,0.959147,0.903010,0.762887,0.443114,0.560606,0.483660,0.842408,0.697534,0.497775
9,XChainWatcher (single-chain comparison),Nomad,fold_5,0.677500,0.841096,0.853081,0.959147,0.903010,0.762887,0.443114,0.560606,0.483660,0.734527,0.607417,0.497775


In [191]:
summary = all_stats.groupby(["model", "incident"])[METRIC_COLS].agg(["mean", "var"])
summary

loss             \
                                                          mean        var   
model                                   incident                            
XChainWatcher (single-chain comparison) Nomad         0.586780   0.054160   
                                        Ronin #1      7.685973  31.894758   
XScope (cross-chain model)              Qubit         5.463814   3.055141   
                                        THORChain #2  1.599658   2.187637   
                                        THORChain #3  0.002249   0.000005   
XScope (single-chain model)             Qubit         6.336176  23.554113   
                                        THORChain #1  0.410290   0.772705   
                                        THORChain #2  7.831581  20.156341   
                                        THORChain #3  6.832929   1.427900   

                                                      accuracy            \
                                                          mean       var   
model                                   incident                           
XChainWatcher (single-chain comparison) Nomad         0.827123  0.000976   
                                        Ronin #1      0.100000  0.050000   
XScope (cross-chain model)              Qubit         0.157895  0.000000   
                                        THORChain #2  0.500000  0.250000   
                                        THORChain #3  1.000000  0.000000   
XScope (single-chain model)             Qubit         0.200000  0.200000   
                                        THORChain #1  0.800000  0.200000   
                                        THORChain #2  0.200000  0.200000   
                                        THORChain #3  0.250000  0.000000   

                                                     precision_normal  \
                                                                 mean   
model                                   incident                        
XChainWatcher (single-chain comparison) Nomad                0.836711   
                                        Ronin #1             0.000000   
XScope (cross-chain model)              Qubit                0.157895   
                                        THORChain #2         0.000000   
                                        THORChain #3         0.000000   
XScope (single-chain model)             Qubit                0.000000   
                                        THORChain #1         0.000000   
                                        THORChain #2         0.000000   
                                        THORChain #3         0.250000   

                                                              recall_normal  \
                                                          var          mean   
model                                   incident                              
XChainWatcher (single-chain comparison) Nomad         0.00134      0.967318   
                                        Ronin #1      0.00000      0.000000   
XScope (cross-chain model)              Qubit         0.00000      1.000000   
                                        THORChain #2  0.00000      0.000000   
                                        THORChain #3  0.00000      0.000000   
XScope (single-chain model)             Qubit         0.00000      0.000000   
                                        THORChain #1  0.00000      0.000000   
                                        THORChain #2  0.00000      0.000000   
                                        THORChain #3  0.00000      1.000000   

                                                               f1_normal  \
                                                           var      mean   
model                                   incident                           
XChainWatcher (single-chain comparison) Nomad         0.000334  0.896577   
                                        Ronin #1      0.000000  0.000000   
XScope (cross-

In [192]:
summary_flat = summary.copy()
summary_flat.columns = [f"{metric}_{stat}" for metric, stat in summary_flat.columns]
summary_flat = summary_flat.reset_index()
summary_flat


,model,incident,loss_mean,loss_var,accuracy_mean,accuracy_var,precision_normal_mean,precision_normal_var,recall_normal_mean,recall_normal_var,...,f1_anomaly_mean,f1_anomaly_var,f2_anomaly_mean,f2_anomaly_var,roc_auc_mean,roc_auc_var,pr_auc_mean,pr_auc_var,mcc_mean,mcc_var
0,XChainWatcher (single-chain comparison),Nomad,0.586780,0.054160,0.827123,0.000976,0.836711,0.00134,0.967318,0.000334,...,0.448485,0.062856,0.386928,0.046785,0.840387,0.006915,0.666605,0.008434,0.39822,0.049556
1,XChainWatcher (single-chain comparison),Ronin #1,7.685973,31.894758,0.100000,0.050000,0.000000,0.00000,0.000000,0.000000,...,0.133333,0.088889,0.111111,0.061728,NaN,NaN,1.000000,0.000000,0.00000,0.000000
2,XScope (cross-chain model),Qubit,5.463814,3.055141,0.157895,0.000000,0.157895,0.00000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.095833,0.045920,0.724891,0.003111,0.00000,0.000000
3,XScope (cross-chain model),THORChain #2,1.599658,2.187637,0.500000,0.250000,0.000000,0.00000,0.000000,0.000000,...,0.533333,0.255556,0.511111,0.250617,NaN,NaN,1.000000,0.000000,0.00000,0.000000
4,XScope (cross-chain model),THORChain #3,0.002249,0.000005,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,1.000000,0.000000,1.000000,0.000000,NaN,NaN,1.000000,0.000000,0.00000,0.000000
5,XScope (single-chain model),Qubit,6.336176,23.554113,0.200000,0.200000,0.000000,0.00000,0.000000,0.000000,...,0.200000,0.200000,0.200000,0.200000,NaN,NaN,1.000000,0.000000,0.00000,0.000000
6,XScope (single-chain model),THORChain #1,0.410290,0.772705,0.800000,0.200000,0.000000,0.00000,0.000000,0.000000,...,0.800000,0.200000,0.800000,0.200000,NaN,NaN,1.000000,0.000000,0.00000,0.000000
7,XScope (single-chain model),THORChain #2,7.831581,20.156341,0.200000,0.200000,0.000000,0.00000,0.000000,0.000000,...,0.200000,0.200000,0.200000,0.200000,NaN,NaN,1.000000,0.000000,0.00000,0.000000
8,XScope (single-chain model),THORChain #3,6.832929,1.427900,0.250000,0.000000,0.250000,0.00000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.200000,0.200000,0.800000,0.012500,0.00000,0.000000


In [193]:
latex_table = summary_flat.round(4).to_latex(
    index=False,
    escape=True,
    caption="Mean and variance of evaluation metrics per test.",
    label="tab:eval-summary",
)
print(latex_table)


\begin{table}
\caption{Mean and variance of evaluation metrics per test.}
\label{tab:eval-summary}
\begin{tabular}{llrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
model & incident & loss\_mean & loss\_var & accuracy\_mean & accuracy\_var & precision\_normal\_mean & precision\_normal\_var & recall\_normal\_mean & recall\_normal\_var & f1\_normal\_mean & f1\_normal\_var & precision\_anomaly\_mean & precision\_anomaly\_var & recall\_anomaly\_mean & recall\_anomaly\_var & f1\_anomaly\_mean & f1\_anomaly\_var & f2\_anomaly\_mean & f2\_anomaly\_var & roc\_auc\_mean & roc\_auc\_var & pr\_auc\_mean & pr\_auc\_var & mcc\_mean & mcc\_var \\
\midrule
XChainWatcher (single-chain comparison) & Nomad & 0.586800 & 0.054200 & 0.827100 & 0.001000 & 0.836700 & 0.001300 & 0.967300 & 0.000300 & 0.896600 & 0.000200 & 0.610300 & 0.116400 & 0.354500 & 0.039300 & 0.448500 & 0.062900 & 0.386900 & 0.046800 & 0.840400 & 0.006900 & 0.666600 & 0.008400 & 0.398200 & 0.049600 \\
XChainWatcher (single-chain comparison) & Ronin 

In [194]:
def fmt_mean_var(mean, var, decimals=4, percent=False):
    if percent:
        mean *= 100
        var *= 100
    suffix = "\\%" if percent else ""
    return f"{mean:05.{decimals}f} ± {var:05.{decimals}f}{suffix}"

combined = summary_flat[["model", "incident"]].copy()
for metric in METRIC_COLS:
    combined[metric] = [
        fmt_mean_var(m, v, decimals=2 if metric not in ["loss"] else 4, percent=True if metric not in ["loss"] else False)
        for m, v in zip(summary_flat[f"{metric}_mean"], summary_flat[f"{metric}_var"])
    ]

def load_dataset_stats(model: str, incident: str, run_dir: Path) -> dict | None:
    params_path = run_dir / "params.yaml"
    if not params_path.exists():
        return None
    with open(params_path) as f:
        params = yaml.safe_load(f)
    stats = params["dataset"]["stats"]
    return {
        "model": model,
        "incident": incident,
        "total_samples": stats["total_samples"],
        "anomaly_samples": stats["anomaly_samples"],
    }

dataset_stats = pd.DataFrame(
    [d for model, incident, path in RUN_ROWS
     if (d := load_dataset_stats(model, incident, path)) is not None]
)

combined = combined.merge(dataset_stats, on=["model", "incident"])
cols = ["model", "incident", "total_samples", "anomaly_samples"] + METRIC_COLS
combined = combined[cols]
combined

,model,incident,total_samples,anomaly_samples,loss,accuracy,precision_normal,recall_normal,f1_normal,precision_anomaly,recall_anomaly,f1_anomaly,f2_anomaly,roc_auc,pr_auc,mcc
0,XChainWatcher (single-chain comparison),Nomad,730,167,0.5868 ± 0.0542,82.71 ± 00.10\%,83.67 ± 00.13\%,96.73 ± 00.03\%,89.66 ± 00.02\%,61.03 ± 11.64\%,35.45 ± 03.93\%,44.85 ± 06.29\%,38.69 ± 04.68\%,84.04 ± 00.69\%,66.66 ± 00.84\%,39.82 ± 04.96\%
1,XChainWatcher (single-chain comparison),Ronin #1,2,2,7.6860 ± 31.8948,10.00 ± 05.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,20.00 ± 20.00\%,10.00 ± 05.00\%,13.33 ± 08.89\%,11.11 ± 06.17\%,00nan ± 00nan\%,100.00 ± 00.00\%,00.00 ± 00.00\%
2,XScope (cross-chain model),Qubit,19,16,5.4638 ± 3.0551,15.79 ± 00.00\%,15.79 ± 00.00\%,100.00 ± 00.00\%,27.27 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,09.58 ± 04.59\%,72.49 ± 00.31\%,00.00 ± 00.00\%
3,XScope (cross-chain model),THORChain #2,2,2,1.5997 ± 2.1876,50.00 ± 25.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,60.00 ± 30.00\%,50.00 ± 25.00\%,53.33 ± 25.56\%,51.11 ± 25.06\%,00nan ± 00nan\%,100.00 ± 00.00\%,00.00 ± 00.00\%
4,XScope (cross-chain model),THORChain #3,6,6,0.0022 ± 0.0000,100.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,100.00 ± 00.00\%,100.00 ± 00.00\%,100.00 ± 00.00\%,100.00 ± 00.00\%,00nan ± 00nan\%,100.00 ± 00.00\%,00.00 ± 00.00\%
5,XScope (single-chain model),Qubit,16,16,6.3362 ± 23.5541,20.00 ± 20.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,20.00 ± 20.00\%,20.00 ± 20.00\%,20.00 ± 20.00\%,20.00 ± 20.00\%,00nan ± 00nan\%,100.00 ± 00.00\%,00.00 ± 00.00\%
6,XScope (single-chain model),THORChain #1,6,6,0.4103 ± 0.7727,80.00 ± 20.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,80.00 ± 20.00\%,80.00 ± 20.00\%,80.00 ± 20.00\%,80.00 ± 20.00\%,00nan ± 00nan\%,100.00 ± 00.00\%,00.00 ± 00.00\%
7,XScope (single-chain model),THORChain #2,15,15,7.8316 ± 20.1563,20.00 ± 20.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,20.00 ± 20.00\%,20.00 ± 20.00\%,20.00 ± 20.00\%,20.00 ± 20.00\%,00nan ± 00nan\%,100.00 ± 00.00\%,00.00 ± 00.00\%
8,XScope (single-chain model),THORChain #3,8,6,6.8329 ± 1.4279,25.00 ± 00.00\%,25.00 ± 00.00\%,100.00 ± 00.00\%,40.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,00.00 ± 00.00\%,20.00 ± 20.00\%,80.00 ± 01.25\%,00.00 ± 00.00\%


In [ ]:
combined_latex_ready = combined.copy()
for metric in METRIC_COLS:
    combined_latex_ready[metric] = combined_latex_ready[metric].str.replace(
        "±", r"$\pm$", regex=False
    )

combined_latex_ready = combined_latex_ready.set_index(["model", "incident"])

latex_combined = combined_latex_ready.to_latex(
    multirow=True,
    escape=False,
    caption="Cross-incident evaluation metrics based on incidents identified by earlier models.",
    label="table:evaluation-cross-eval-incidents",
)
print(latex_combined)
# NOTE: Requires `multirow` package in LaTeX

\begin{table}
\caption{Mean and variance (mean $\pm$ var) of evaluation metrics per incident, grouped by model.}
\label{tab:eval-summary-incidents}
\begin{tabular}{llrrllllllllllll}
\toprule
 &  & total_samples & anomaly_samples & loss & accuracy & precision_normal & recall_normal & f1_normal & precision_anomaly & recall_anomaly & f1_anomaly & f2_anomaly & roc_auc & pr_auc & mcc \\
model & incident &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{XChainWatcher (single-chain comparison)} & Nomad & 730 & 167 & 0.5868 $\pm$ 0.0542 & 82.71 $\pm$ 00.10\% & 83.67 $\pm$ 00.13\% & 96.73 $\pm$ 00.03\% & 89.66 $\pm$ 00.02\% & 61.03 $\pm$ 11.64\% & 35.45 $\pm$ 03.93\% & 44.85 $\pm$ 06.29\% & 38.69 $\pm$ 04.68\% & 84.04 $\pm$ 00.69\% & 66.66 $\pm$ 00.84\% & 39.82 $\pm$ 04.96\% \\
 & Ronin #1 & 2 & 2 & 7.6860 $\pm$ 31.8948 & 10.00 $\pm$ 05.00\% & 00.00 $\pm$ 00.00\% & 00.00 $\pm$ 00.00\% & 00.00 $\pm$ 00.00\% & 20.00 $\pm$ 20.00\% & 10.00 $\pm$ 05.00\% & 13.33 $\pm$ 08.89\% 